In [17]:
import torch
import os
from sklearn.metrics import confusion_matrix, f1_score, classification_report

# Adjust import paths if needed
from src.Project.components.model_hfstream import SelfDrivingModel
from src.Project.components.dataset_class_hfstream import SelfDrivingDataset
from torch.utils.data import DataLoader

In [19]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

project_root = r"C:\Users\Nandita\Downloads\End-to-End-Deep-Learning-for-Autonomous-Driving"
val_data_path = r'C:\Users\Nandita\Downloads\End-to-End-Deep-Learning-for-Autonomous-Driving\data\processed_json\val.json'
model_path = r'C:\Users\Nandita\Downloads\End-to-End-Deep-Learning-for-Autonomous-Driving\artifacts\checkpoints\best_model.pt'

dataset = SelfDrivingDataset(val_data_path, project_root)
dataloader = DataLoader(dataset, batch_size=4, shuffle=False)

model = SelfDrivingModel().to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

SelfDrivingModel(
  (rgb_encoder): ImageEncoder(
    (net): Sequential(
      (0): ConvBlock(
        (conv): Sequential(
          (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
      )
      (1): ConvBlock(
        (conv): Sequential(
          (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
      )
      (2): ConvBlock(
        (conv): Sequential(
          (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
      )
      (3): ConvBlock(
        (conv): Sequential(
          (0): Conv2d(128, 256, kernel_size

In [21]:
all_preds = []
all_targets = []

with torch.no_grad():
    for batch in dataloader:

        front = batch["front"].to(device)
        left = batch["left"].to(device)
        right = batch["right"].to(device)
        seg = batch["seg"].to(device)
        state = batch["state"].to(device)
        target = batch["target"].to(device)

        outputs = model(front, left, right, seg, state)

        brake_pred = outputs[:, 2]
        brake_true = target[:, 2]

        brake_pred_class = (brake_pred >= 0.5).int()
        brake_true_class = (brake_true >= 0.5).int()

        all_preds.extend(brake_pred_class.cpu().numpy())
        all_targets.extend(brake_true_class.cpu().numpy())

cm = confusion_matrix(all_targets, all_preds)
f1 = f1_score(all_targets, all_preds)

print('Confusion Matrix (Brake):')
print(cm)

print('\nF1 Score (Brake):', f1)

print('\nClassification Report:')
print(classification_report(all_targets, all_preds))


Confusion Matrix (Brake):
[[  0 894]
 [  0 604]]

F1 Score (Brake): 0.5746907706945766

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       894
           1       0.40      1.00      0.57       604

    accuracy                           0.40      1498
   macro avg       0.20      0.50      0.29      1498
weighted avg       0.16      0.40      0.23      1498



C:\Users\Nandita\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Nandita\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Nandita\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [13]:
all_brake_values = []

with torch.no_grad():
    for batch in dataloader:
        front = batch["front"].to(device)
        left = batch["left"].to(device)
        right = batch["right"].to(device)
        seg = batch["seg"].to(device)
        state = batch["state"].to(device)

        outputs = model(front, left, right, seg, state)
        brake_pred = outputs[:, 2]

        all_brake_values.extend(brake_pred.cpu().numpy())

print("Min:", min(all_brake_values))
print("Max:", max(all_brake_values))
print("Mean:", sum(all_brake_values)/len(all_brake_values))

Min: 0.9998734
Max: 0.9999974
Mean: 0.9999640175751914


In [15]:
all_true_brake = []

for batch in dataloader:
    target = batch["target"]
    brake_true = target[:, 2]
    all_true_brake.extend(brake_true.numpy())

print("True Min:", min(all_true_brake))
print("True Max:", max(all_true_brake))
print("True Mean:", sum(all_true_brake)/len(all_true_brake))

True Min: 0.0
True Max: 1.0
True Mean: 0.40557408502303033
